In [1]:
import pandas as pd
import numpy as np
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2011_2012"
    / "Questionnnair"
)

xpt_file = DATA_DIR / "HOQ_G.xpt"
csv_file = DATA_DIR / "HOQ_G_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

#
# Preserve untouched pandas-decoded XPT.
#
source_df = df.copy()


print("=" * 80)
print("NHANES 2011-2012 HOQ_G XPT -> CSV")
print("=" * 80)

print(
    "Rows:",
    f"{len(df):,}"
)

print(
    "Columns:",
    len(df.columns)
)


# ============================================================
# 3. EXPECTED STRUCTURE
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "HOD050",
    "HOQ065"
]


if df.columns.tolist() != EXPECTED_COLUMNS:

    raise ValueError(
        "HOQ_G column names/order differ from expected.\n"
        f"Expected: {EXPECTED_COLUMNS}\n"
        f"Actual:   {df.columns.tolist()}"
    )


if len(df) != 9756:

    raise ValueError(
        f"Unexpected row count: {len(df):,}"
    )


if len(df.columns) != 3:

    raise ValueError(
        f"Unexpected column count: {len(df.columns)}"
    )


print(
    "PASS: Structure = 9,756 rows x 3 columns."
)


# ============================================================
# 4. CHECK KNOWN XPORT TINY-ZERO ARTIFACT
# ============================================================

#
# Known pandas/SAS XPORT zero-decoding artifact:
#
#     5.397605346934028e-79
#
# Attached HOQ_G.xpt:
#
#     tiny artifacts = 0
#
# Therefore NO zero restoration is needed.
#
# IMPORTANT:
#
# If a different/local copy contains tiny values,
# STOP rather than globally changing them to zero.
#

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


numeric_cols = (
    df.select_dtypes(
        include=[np.number]
    )
    .columns
    .tolist()
)


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)


tiny_total = int(
    tiny_counts.sum()
)


true_zero_counts = (
    df[numeric_cols]
    .eq(0)
    .sum()
)


true_zero_total = int(
    true_zero_counts.sum()
)


print("\n--- ORIGINAL XPT ZERO CHECK ---")


print(
    "Ordinary numeric zeros:",
    f"{true_zero_total:,}"
)


print(
    "Tiny XPORT artifacts:",
    f"{tiny_total:,}"
)


if tiny_total > 0:

    print(
        "\nTiny values by variable:"
    )

    print(
        tiny_counts[
            tiny_counts > 0
        ]
        .sort_values(
            ascending=False
        )
        .to_string()
    )


    raise ValueError(
        "STOP: Unexpected tiny XPORT artifacts "
        "found in HOQ_G. Do not automatically "
        "replace them with zero."
    )


if true_zero_total > 0:

    print(
        "\nTrue zeros by variable:"
    )

    print(
        true_zero_counts[
            true_zero_counts > 0
        ]
        .sort_values(
            ascending=False
        )
        .to_string()
    )


    raise ValueError(
        "STOP: Unexpected numeric zeros found "
        "in HOQ_G."
    )


print(
    "PASS: No tiny XPORT artifacts."
)

print(
    "PASS: No ordinary numeric zeros."
)


# ============================================================
# 5. CHECK FOR GENUINE DECIMAL VARIABLES
# ============================================================

#
# Always perform this BEFORE Int64 conversion.
#
# Attached HOQ_G:
#
#     genuine fractional variables = NONE
#
# All three fields are identifier / whole-number
# questionnaire values.
#
# If decimals unexpectedly appear, STOP instead of
# rounding them.
#

print("\n--- FRACTIONAL VALUE CHECK ---")


fractional_columns = []


for col in numeric_cols:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) == 0:
        continue


    fractional_mask = ~np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )


    fractional_count = int(
        fractional_mask.sum()
    )


    print(
        f"{col}: "
        f"{fractional_count:,} fractional observations"
    )


    if fractional_count > 0:

        fractional_columns.append(
            col
        )


if fractional_columns:

    raise ValueError(
        "STOP: Genuine decimal values unexpectedly "
        "found in HOQ_G. They will NOT be rounded:\n"
        + str(fractional_columns)
    )


print(
    "PASS: No genuine fractional variables."
)


# ============================================================
# 6. CLEAN AND VALIDATE SEQN
# ============================================================

seqn = (
    df["SEQN"]
    .dropna()
)


if not np.all(
    np.isclose(
        seqn,
        np.round(seqn),
        rtol=0,
        atol=1e-12
    )
):

    raise ValueError(
        "SEQN contains unexpected fractional values."
    )


df["SEQN"] = (
    df["SEQN"]
    .astype("Int64")
)


print("\n--- SEQN ---")


print(
    "Minimum:",
    df["SEQN"].min()
)


print(
    "Maximum:",
    df["SEQN"].max()
)


print(
    "Missing:",
    int(
        df["SEQN"]
        .isna()
        .sum()
    )
)


print(
    "Duplicates:",
    int(
        df["SEQN"]
        .duplicated()
        .sum()
    )
)


if not (
    df["SEQN"].min() == 62161
    and
    df["SEQN"].max() == 71916
    and
    int(
        df["SEQN"]
        .isna()
        .sum()
    ) == 0
    and
    int(
        df["SEQN"]
        .duplicated()
        .sum()
    ) == 0
):

    raise ValueError(
        "SEQN validation failed."
    )


print(
    "PASS: SEQN validated."
)


# ============================================================
# 7. FORMAT ALL PROVEN WHOLE-NUMBER VARIABLES
# ============================================================

#
# HOQ_G contains NO genuine decimal variables.
#
# Therefore all three fields can safely become
# nullable Int64.
#
# IMPORTANT:
#
# NO .round() is used for conversion.
#

INTEGER_COLUMNS = EXPECTED_COLUMNS.copy()


for col in INTEGER_COLUMNS:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) > 0:

        whole_mask = np.isclose(
            values,
            np.round(values),
            rtol=0,
            atol=1e-12
        )


        if not whole_mask.all():

            raise ValueError(
                f"{col} contains genuine decimal values "
                "and cannot safely be converted to Int64."
            )


    df[col] = (
        df[col]
        .astype("Int64")
    )


print(
    "\nPASS: All 3 HOQ_G variables converted "
    "to nullable Int64 without rounding."
)


# ============================================================
# 8. CDC CHECK - HOD050
# ============================================================

#
# HOD050 = Number of rooms in home
#
# CDC:
#
# 1      63
# 2     241
# 3     924
# 4    1863
# 5    1972
# 6    1709
# 7    1117
# 8     776
# 9     410
# 10    309
# 11    162
# 12     67
# 13     90
# 777     4
# 999     0
# Missing 49
#
# IMPORTANT:
#
# CDC states values >=13 are grouped and
# released as 13.
#

EXPECTED_HOD050 = {
    1: 63,
    2: 241,
    3: 924,
    4: 1863,
    5: 1972,
    6: 1709,
    7: 1117,
    8: 776,
    9: 410,
    10: 309,
    11: 162,
    12: 67,
    13: 90,
    777: 4
}


actual_hod050 = (
    df["HOD050"]
    .value_counts()
    .sort_index()
    .to_dict()
)


if actual_hod050 != EXPECTED_HOD050:

    raise ValueError(
        "HOD050 frequencies differ from CDC.\n"
        f"Expected: {EXPECTED_HOD050}\n"
        f"Actual:   {actual_hod050}"
    )


if int(
    df["HOD050"]
    .isna()
    .sum()
) != 49:

    raise ValueError(
        "HOD050 missing count differs from CDC."
    )


if int(
    df["HOD050"]
    .eq(999)
    .sum()
) != 0:

    raise ValueError(
        "Unexpected HOD050=999 value found."
    )


print(
    "PASS: HOD050 matches CDC exactly."
)


# ============================================================
# 9. HOD050 RANGE / TOP-CODE CHECK
# ============================================================

#
# Valid released room counts:
#
#     1-13
#
# 13 means:
#
#     13 OR MORE
#
# Special code:
#
#     777 = Refused
#

hod_valid = (
    df["HOD050"]
    .between(
        1,
        13
    )
)


if int(
    hod_valid.sum()
) != 9703:

    raise ValueError(
        "Unexpected HOD050 valid room-count total."
    )


if int(
    df["HOD050"]
    .eq(13)
    .sum()
) != 90:

    raise ValueError(
        "Unexpected HOD050 top-coded 13 count."
    )


print(
    "PASS: HOD050 top-coded 13-or-more "
    "category validated."
)


# ============================================================
# 10. CDC CHECK - HOQ065
# ============================================================

#
# HOQ065 = Home owned, bought, rented, other
#
# CDC:
#
# 1 = Owned or being bought    5003
# 2 = Rented                   4434
# 3 = Other arrangement         259
# 7 = Refused                     9
# 9 = Don't know                  2
# Missing                        49
#

EXPECTED_HOQ065 = {
    1: 5003,
    2: 4434,
    3: 259,
    7: 9,
    9: 2
}


actual_hoq065 = (
    df["HOQ065"]
    .value_counts()
    .sort_index()
    .to_dict()
)


if actual_hoq065 != EXPECTED_HOQ065:

    raise ValueError(
        "HOQ065 frequencies differ from CDC.\n"
        f"Expected: {EXPECTED_HOQ065}\n"
        f"Actual:   {actual_hoq065}"
    )


if int(
    df["HOQ065"]
    .isna()
    .sum()
) != 49:

    raise ValueError(
        "HOQ065 missing count differs from CDC."
    )


print(
    "PASS: HOQ065 matches CDC exactly."
)


# ============================================================
# 11. ALLOWED RELEASED CODE CHECK
# ============================================================

#
# Additional protection against accidental corruption.
#

HOD050_ALLOWED = (
    set(
        range(
            1,
            14
        )
    )
    |
    {
        777
    }
)


HOQ065_ALLOWED = {
    1,
    2,
    3,
    7,
    9
}


observed_hod050 = set(
    df["HOD050"]
    .dropna()
    .astype(int)
    .unique()
    .tolist()
)


observed_hoq065 = set(
    df["HOQ065"]
    .dropna()
    .astype(int)
    .unique()
    .tolist()
)


unexpected_hod050 = (
    observed_hod050
    -
    HOD050_ALLOWED
)


unexpected_hoq065 = (
    observed_hoq065
    -
    HOQ065_ALLOWED
)


if unexpected_hod050:

    raise ValueError(
        "Unexpected HOD050 values: "
        f"{sorted(unexpected_hod050)}"
    )


if unexpected_hoq065:

    raise ValueError(
        "Unexpected HOQ065 values: "
        f"{sorted(unexpected_hoq065)}"
    )


print(
    "PASS: HOD050 and HOQ065 contain "
    "only expected released values."
)


# ============================================================
# 12. HOUSEHOLD MISSING-PATTERN CHECK
# ============================================================

#
# HOD050 and HOQ065 each have 49 missing records.
#
# In the attached XPT, those missing positions are
# exactly the same.
#

same_missing_pattern = np.array_equal(

    df["HOD050"]
    .isna()
    .to_numpy(),

    df["HOQ065"]
    .isna()
    .to_numpy()
)


if not same_missing_pattern:

    raise ValueError(
        "HOD050 and HOQ065 missing-position "
        "patterns differ unexpectedly."
    )


print(
    "PASS: HOD050 and HOQ065 have the same "
    "49 missing household records."
)


# ============================================================
# 13. VALUE FIDELITY BEFORE EXPORT
# ============================================================

#
# HOQ_G requires ZERO numeric corrections.
#
# Therefore every value must remain numerically
# identical to the original XPT.
#
# Only textual representation changes:
#
#     SEQN:
#       62161.0 -> 62161
#
#     HOD050:
#       6.0 -> 6
#
#     HOQ065:
#       2.0 -> 2
#

print("\n--- VALUE FIDELITY BEFORE EXPORT ---")


fidelity_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        source_df[col]
        .to_numpy(
            dtype=float
        )
    )


    current_values = (
        pd.to_numeric(
            df[col],
            errors="coerce"
        )
        .to_numpy(
            dtype=float,
            na_value=np.nan
        )
    )


    same = np.allclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        fidelity_failures.append(
            col
        )


if fidelity_failures:

    raise ValueError(
        "Unexpected value changes before export:\n"
        + str(fidelity_failures)
    )


print(
    "PASS: All 3 variables preserve "
    "source XPT values exactly."
)


# ============================================================
# 14. EXPORT CSV
# ============================================================

#
# No rounding.
#
# No:
#
#     float_format=
#
# Missing values become blank.
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print("\nCSV created:")

print(
    csv_file
)


# ============================================================
# 15. RE-READ CSV WITH ROUND-TRIP PARSER
# ============================================================

roundtrip_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


if roundtrip_df.shape != df.shape:

    raise ValueError(
        "CSV dimensions changed."
    )


if (
    roundtrip_df.columns.tolist()
    != EXPECTED_COLUMNS
):

    raise ValueError(
        "CSV column names/order changed."
    )


print(
    "PASS: CSV structure preserved."
)


# ============================================================
# 16. EXACT XPT -> CSV NUMERIC FIDELITY
# ============================================================

print("\n--- EXACT XPT -> CSV FIDELITY ---")


csv_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        source_df[col]
        .to_numpy(
            dtype=float
        )
    )


    exported_values = (
        roundtrip_df[col]
        .to_numpy(
            dtype=float
        )
    )


    same = np.allclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        csv_failures.append(
            col
        )


if csv_failures:

    raise ValueError(
        "XPT -> CSV numeric differences found:\n"
        + str(csv_failures)
    )


print(
    "PASS: All 3 CSV variables round-trip "
    "to exact source XPT numeric values."
)


# ============================================================
# 17. MISSING VALUE FIDELITY
# ============================================================

missing_mismatches = 0


for col in EXPECTED_COLUMNS:

    expected_missing = (
        source_df[col]
        .isna()
        .to_numpy()
    )


    exported_missing = (
        roundtrip_df[col]
        .isna()
        .to_numpy()
    )


    missing_mismatches += int(
        np.sum(
            expected_missing
            !=
            exported_missing
        )
    )


print("\n--- MISSING VALUE FIDELITY ---")


print(
    "Missing-position mismatches:",
    missing_mismatches
)


if missing_mismatches != 0:

    raise ValueError(
        "Missing-value positions changed."
    )


print(
    "PASS: Missing values preserved exactly."
)


# ============================================================
# 18. READ RAW CSV AS TEXT
# ============================================================

raw_csv_df = pd.read_csv(
    csv_file,
    dtype=str,
    keep_default_na=False
)


# ============================================================
# 19. FINAL TINY-VALUE CHECK
# ============================================================

tiny_csv_count = 0


for col in EXPECTED_COLUMNS:

    numeric_values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )


    tiny_csv_count += int(
        numeric_values
        .eq(TINY_VALUE)
        .sum()
    )


print("\n--- FINAL TINY VALUE CHECK ---")


print(
    "Tiny values remaining:",
    tiny_csv_count
)


if tiny_csv_count != 0:

    raise ValueError(
        "Tiny XPORT artifacts found "
        "in final CSV."
    )


print(
    "PASS: No tiny XPORT artifacts."
)


# ============================================================
# 20. RAW CSV .0 FORMAT CHECK
# ============================================================

#
# HOQ_G contains NO genuine decimal variables.
#
# Therefore no populated value should end in ".0".
#
# This global rule is safe for HOQ_G specifically.
#
# DO NOT apply this globally to Dietary or any
# Lab/Exam/Questionnaire dataset containing genuine
# decimal measurements.
#

print("\n--- RAW CSV .0 CHECK ---")


dot_zero_errors = {}


for col in INTEGER_COLUMNS:

    tokens = raw_csv_df[col]


    bad_mask = (
        tokens.ne("")
        &
        tokens.str.endswith(".0")
    )


    count = int(
        bad_mask.sum()
    )


    if count > 0:

        dot_zero_errors[col] = count


if dot_zero_errors:

    raise ValueError(
        "Unexpected .0 formatting found:\n"
        + str(dot_zero_errors)
    )


print(
    "PASS: No unnecessary .0 formatting."
)


# ============================================================
# 21. SCIENTIFIC NOTATION CHECK
# ============================================================

with open(
    csv_file,
    "r",
    encoding="utf-8"
) as f:

    csv_text = f.read()


scientific_count = (
    csv_text.lower().count("e-")
    +
    csv_text.lower().count("e+")
)


print(
    "\nScientific notation occurrences:",
    scientific_count
)


if scientific_count != 0:

    raise ValueError(
        "Unexpected scientific notation found."
    )


print(
    "PASS: No scientific notation."
)


# ============================================================
# 22. FINAL SEQN CHECK
# ============================================================

final_seqn = pd.to_numeric(
    raw_csv_df["SEQN"],
    errors="raise"
)


if not (
    final_seqn.min() == 62161
    and
    final_seqn.max() == 71916
    and
    int(
        final_seqn.duplicated()
        .sum()
    ) == 0
):

    raise ValueError(
        "Final CSV SEQN validation failed."
    )


if (
    raw_csv_df["SEQN"]
    .str.endswith(".0")
    .any()
):

    raise ValueError(
        "Final CSV SEQN still contains .0."
    )


print(
    "PASS: Final CSV SEQN valid "
    "and contains no .0."
)


# ============================================================
# 23. FINAL SUMMARY
# ============================================================

print(
    "\n"
    + "=" * 80
)


print(
    "HOQ_G CONVERSION COMPLETE"
)


print(
    "=" * 80
)


print(
    "Rows:",
    f"{len(df):,}"
)


print(
    "Columns:",
    len(df.columns)
)


print(
    "Original tiny-value artifacts:",
    tiny_total
)


print(
    "Original true numeric zeros:",
    true_zero_total
)


print(
    "Validated zero restorations: NONE"
)


print(
    "Genuine decimal variables: NONE"
)


print(
    "Whole-number variables:",
    len(INTEGER_COLUMNS)
)


print(
    "HOD050 valid room-count records:",
    int(
        df["HOD050"]
        .between(
            1,
            13
        )
        .sum()
    )
)


print(
    "HOD050 top-coded 13-or-more records:",
    int(
        df["HOD050"]
        .eq(13)
        .sum()
    )
)


print(
    "XPT -> CSV unexpected numeric differences: "
    "0 expected"
)


print(
    "Missing-position differences: "
    "0 expected"
)


print(
    "Tiny values remaining: "
    "0 expected"
)


print(
    "Unnecessary .0: "
    "0 expected"
)


print(
    "Scientific notation: "
    "0 expected"
)

NHANES 2011-2012 HOQ_G XPT -> CSV
Rows: 9,756
Columns: 3
PASS: Structure = 9,756 rows x 3 columns.

--- ORIGINAL XPT ZERO CHECK ---
Ordinary numeric zeros: 0
Tiny XPORT artifacts: 0
PASS: No tiny XPORT artifacts.
PASS: No ordinary numeric zeros.

--- FRACTIONAL VALUE CHECK ---
SEQN: 0 fractional observations
HOD050: 0 fractional observations
HOQ065: 0 fractional observations
PASS: No genuine fractional variables.

--- SEQN ---
Minimum: 62161
Maximum: 71916
Missing: 0
Duplicates: 0
PASS: SEQN validated.

PASS: All 3 HOQ_G variables converted to nullable Int64 without rounding.
PASS: HOD050 matches CDC exactly.
PASS: HOD050 top-coded 13-or-more category validated.
PASS: HOQ065 matches CDC exactly.
PASS: HOD050 and HOQ065 contain only expected released values.
PASS: HOD050 and HOQ065 have the same 49 missing household records.

--- VALUE FIDELITY BEFORE EXPORT ---
PASS: All 3 variables preserve source XPT values exactly.

CSV created:
C:\Users\User\Desktop\Projects\Clinical_Neighbourhood_